In [ ]:
import os
import glob
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from google.colab import files

csv_files = glob.glob('*.csv')
if not csv_files:
    uploaded = files.upload()
    csv_files = list(uploaded.keys())

df = pd.read_csv(csv_files[0]).dropna()
if len(df) > 200000:
    df = df.sample(n=200000, random_state=42).reset_index(drop=True)

rename_map = {'vp_7000m': 'Vp_7000m', 'vp_6000m': 'Vp_6000m', 'vpvs_60km': 'VpVs_60km', 'dvpvs_60km': 'dVpVs_60km'}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

fitur_terbaik = [
    'Vp_7000m', 'crustal_thickness', 'dVs_20km', 'Vp_6000m', 'VpVs_60km',
    'dVs_60km', 'gg_mantle_60km', 'dVpVs_60km', 'dVs_40km', 'elevation_detrend'
]
target_cols = ['T_1000m', 'T_2000m', 'T_3000m', 'T_4000m', 'T_5000m']

X_raw = df[fitur_terbaik].values
Y = df[target_cols].values

scaler_full = StandardScaler()
X_scaled_full = scaler_full.fit_transform(X_raw)

best_p = {
    'rf_n_estimators': 100,
    'rf_max_depth': 20,
    'xgb_n_estimators': 300,
    'xgb_max_depth': 10,
    'xgb_learning_rate': 0.17332572893079348
}

print('Melatih model_geotermal_hybrid.pkl dan scaler_geotermal.pkl...')
estimators_base = [
    ('rf', RandomForestRegressor(
        n_estimators=best_p['rf_n_estimators'],
        max_depth=best_p['rf_max_depth'],
        random_state=42,
        n_jobs=-1
    )),
    ('xgb', XGBRegressor(
        n_estimators=best_p['xgb_n_estimators'],
        max_depth=best_p['xgb_max_depth'],
        learning_rate=best_p['xgb_learning_rate'],
        random_state=42,
        n_jobs=-1
    ))
]

model_stack = MultiOutputRegressor(
    StackingRegressor(estimators=estimators_base, final_estimator=LinearRegression(), cv=3, n_jobs=-1)
)
model_stack.fit(X_scaled_full, Y)

joblib.dump(model_stack, 'model_geotermal_hybrid.pkl', compress=3)
joblib.dump(scaler_full, 'scaler_geotermal.pkl')
print('Model dan scaler berhasil disimpan.')

files.download('model_geotermal_hybrid.pkl')
files.download('scaler_geotermal.pkl')

Melatih model_geotermal_hybrid.pkl dan scaler_geotermal.pkl...


/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Model dan scaler berhasil disimpan.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>